# 04 — Feature Engineering (PySpark)

## Objective
Transform four raw tables into a single modeling-ready
dataset — one row per approved loan, ~40 features —
capturing applicant risk profile, loan characteristics,
repayment behavioral trajectory, and the Basel III
Expected Loss components.

## The Core Transformation
Input:
  applicant_profile : 50,250 rows × 17 columns
  loan_applications : 50,000 rows × 15 columns
  repayment_history : 813,692 rows × 12 columns
  default_labels    : 27,831 rows × 11 columns

Output:
  modeling_data.csv : 27,831 rows × ~40 features
  (one row per approved loan)

## Feature Categories
1. Static applicant features   — profile at application time
2. Loan characteristics        — amount, rate, tenure, type
3. Repayment behavior aggregate— DPD, missed payments, on-time rate
4. DPD trajectory features     — early vs late window comparison
5. Engineered ratios           — EMI burden, loan-to-income
6. Expected Loss components    — PD score, LGD, EAD from labels
7. Encoding                    — one-hot, target encoding

## Critical Leakage Warning
default_month, ead, recovery_amount, lgd from default_labels
are computed FROM the default outcome — they cannot be used
as features. Only pd_score (model's own estimate) and
loan_amount are safe to use from that table.

## Two Target Variables
is_defaulted    — binary classification (primary)
expected_loss   — regression (secondary)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings("ignore")

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ["PATH"]

JAR_PATH   = os.path.abspath("../jars/sqlite-jdbc-3.45.1.0.jar")
DB_PATH    = os.path.abspath("../data/loan_default.db")
DB_URL     = f"jdbc:sqlite:{DB_PATH}"
PROCESSED  = "../data/processed"
LOCAL_TEMP = os.path.abspath("../data/spark_temp")

os.makedirs(PROCESSED,  exist_ok=True)
os.makedirs(LOCAL_TEMP, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("LoanDefault_FeatureEngineering")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.extraClassPath", JAR_PATH)
    .config("spark.executor.extraClassPath", JAR_PATH)
    .config("spark.local.dir", LOCAL_TEMP)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

PROCESSED = "../data/processed"

profiles  = spark.read.option("header","true") \
                .option("inferSchema","true") \
                .csv(f"{PROCESSED}/profiles_clean.csv")
loans     = spark.read.option("header","true") \
                .option("inferSchema","true") \
                .csv(f"{PROCESSED}/loans_clean.csv")
labels    = spark.read.option("header","true") \
                .option("inferSchema","true") \
                .csv(f"{PROCESSED}/labels_clean.csv")
repayment = spark.read.option("header","true") \
                .option("inferSchema","true") \
                .csv(f"{PROCESSED}/repayment_clean.csv")

loans = loans.withColumn(
    "application_date",
    F.to_date(F.col("application_date"), "yyyy-MM-dd")
)

profiles.createOrReplaceTempView("profiles")
loans.createOrReplaceTempView("loans")
labels.createOrReplaceTempView("labels")
repayment.createOrReplaceTempView("repayment")

repayment.cache()
_ = repayment.count()

print("Data loaded ✅")
print(f"  profiles  : {profiles.count():>8,}")
print(f"  loans     : {loans.count():>8,}")
print(f"  labels    : {labels.count():>8,}")
print(f"  repayment : {repayment.count():>8,} (cached)")

Data loaded ✅
  profiles  :   50,000
  loans     :   50,000
  labels    :   27,831
  repayment :  813,692 (cached)


In [2]:
print("CATEGORY 1 — STATIC APPLICANT FEATURES\n")

applicant_features = profiles.select(
    "customer_id",
    "age",
    "gender",
    "city_tier",
    "employment_type",
    "monthly_income",
    "debt_to_income_ratio",
    "cibil_score",
    "years_employed",
    "number_of_existing_loans",
    "credit_history_years",
    "has_credit_history",
    F.col("has_credit_card").cast(IntegerType()),
    F.col("has_co_applicant").cast(IntegerType()),
    F.col("property_owned").cast(IntegerType()),
)

applicant_features.createOrReplaceTempView("applicant_features")

print(f"Applicant features: {applicant_features.count():,} rows "
      f"× {len(applicant_features.columns)} columns")

# Null check
applicant_features.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in ["cibil_score","monthly_income",
              "debt_to_income_ratio","age"]
]).show()

CATEGORY 1 — STATIC APPLICANT FEATURES

Applicant features: 50,000 rows × 15 columns
+-----------+--------------+--------------------+---+
|cibil_score|monthly_income|debt_to_income_ratio|age|
+-----------+--------------+--------------------+---+
|          0|             0|                   0|  0|
+-----------+--------------+--------------------+---+



In [3]:
print("CATEGORY 2 — LOAN CHARACTERISTICS\n")

loan_features = spark.sql("""
    SELECT
        lo.loan_id,
        lo.customer_id,
        lo.loan_type,
        lo.loan_amount_requested,
        lo.loan_amount_sanctioned,
        lo.tenure_months,
        lo.interest_rate,
        lo.emi_amount,
        COALESCE(lo.ltv_ratio, 0.0)          AS ltv_ratio,
        lo.is_secured,
        lo.application_year,
        lo.application_quarter,

        -- Sanction ratio: how much of requested was approved
        ROUND(COALESCE(
            lo.loan_amount_sanctioned /
            (lo.loan_amount_requested + 1), 0
        ), 4)                                AS sanction_ratio,

        -- EMI to income burden
        ROUND(lo.emi_amount /
              (p.monthly_income + 1), 4)    AS emi_to_income,

        -- Loan to annual income ratio
        ROUND(lo.loan_amount_requested /
              (p.monthly_income * 12 + 1), 4) AS loan_to_annual_income,

        -- Total obligation after this loan
        ROUND((p.monthly_obligations + lo.emi_amount) /
              (p.monthly_income + 1), 4)    AS total_obligation_ratio

    FROM loans lo
    JOIN profiles p ON lo.customer_id = p.customer_id
    WHERE lo.application_status = 'approved'
""")

loan_features.createOrReplaceTempView("loan_features")

print(f"Loan features: {loan_features.count():,} rows "
      f"× {len(loan_features.columns)} columns")

print("\nNew engineered ratios:")
loan_features.select(
    F.round(F.avg("emi_to_income"),4).alias("avg_emi_income"),
    F.round(F.avg("loan_to_annual_income"),4).alias("avg_loan_income"),
    F.round(F.avg("sanction_ratio"),4).alias("avg_sanction_ratio"),
    F.round(F.avg("total_obligation_ratio"),4).alias("avg_total_oblig"),
).show()

CATEGORY 2 — LOAN CHARACTERISTICS

Loan features: 27,831 rows × 16 columns

New engineered ratios:
+--------------+---------------+------------------+---------------+
|avg_emi_income|avg_loan_income|avg_sanction_ratio|avg_total_oblig|
+--------------+---------------+------------------+---------------+
|        0.1296|         0.6488|            0.9001|         0.3969|
+--------------+---------------+------------------+---------------+



In [ ]:
print("CATEGORY 3 — REPAYMENT BEHAVIOR AGGREGATES\n")
print("EDA confirmed these are the strongest predictors")
print("max_dpd: 0.9421, avg_dpd: 0.8606, on_time_rate: 0.6333\n")

repayment_agg = spark.sql("""
    SELECT
        loan_id,

        -- DPD features
        ROUND(AVG(days_past_due), 4)          AS avg_dpd,
        MAX(days_past_due)                    AS max_dpd,
        ROUND(STDDEV(days_past_due), 4)       AS std_dpd,

        -- Payment behavior counts
        COUNT(*)                              AS total_payments,
        SUM(CASE WHEN payment_status = 'missed'
                 THEN 1 ELSE 0 END)           AS missed_count,
        SUM(CASE WHEN days_past_due > 0
                 THEN 1 ELSE 0 END)           AS late_count,
        SUM(CASE WHEN payment_status = 'on_time'
                 THEN 1 ELSE 0 END)           AS on_time_count,
        SUM(CASE WHEN payment_status = 'partial'
                 THEN 1 ELSE 0 END)           AS partial_count,

        -- Payment rate features
        ROUND(AVG(CASE WHEN payment_status = 'on_time'
                       THEN 1.0 ELSE 0 END), 4) AS on_time_rate,
        ROUND(AVG(CASE WHEN payment_status = 'missed'
                       THEN 1.0 ELSE 0 END), 4) AS missed_rate,
        ROUND(AVG(CASE WHEN days_past_due > 0
                       THEN 1.0 ELSE 0 END), 4) AS late_rate,

        -- Outstanding principal trajectory
        ROUND(FIRST(outstanding_principal), 2) AS initial_outstanding,
        ROUND(LAST(outstanding_principal), 2)  AS final_outstanding,

        -- SMA classification features
        -- SMA-0: 1-30 DPD, SMA-1: 31-60 DPD, SMA-2: 61-90 DPD
        SUM(CASE WHEN days_past_due BETWEEN 1  AND 30
                 THEN 1 ELSE 0 END)           AS sma0_count,
        SUM(CASE WHEN days_past_due BETWEEN 31 AND 60
                 THEN 1 ELSE 0 END)           AS sma1_count,
        SUM(CASE WHEN days_past_due BETWEEN 61 AND 89
                 THEN 1 ELSE 0 END)           AS sma2_count

    FROM repayment
    GROUP BY loan_id
""")

repayment_agg.createOrReplaceTempView("repayment_agg")

print(f"Repayment aggregates: {repayment_agg.count():,} rows "
      f"× {len(repayment_agg.columns)} columns")

print("\nBehavior comparison — defaulted vs non-defaulted:")
repayment_agg.join(
    labels.select("loan_id","is_defaulted"), on="loan_id"
).groupBy("is_defaulted").agg(
    F.round(F.avg("avg_dpd"),2).alias("avg_dpd"),
    F.round(F.avg("max_dpd"),1).alias("avg_max_dpd"),
    F.round(F.avg("on_time_rate"),3).alias("avg_on_time_rate"),
    F.round(F.avg("missed_count"),2).alias("avg_missed"),
    F.round(F.avg("sma2_count"),2).alias("avg_sma2"),
).show()

In [ ]:
print("CATEGORY 4 — DPD TRAJECTORY FEATURES\n")
print("Early window (months 1-6) vs late window (months 7-12)")
print("This captures whether payment behavior is worsening\n")

dpd_trajectory = spark.sql("""
    SELECT
        loan_id,

        -- Early window DPD (months 1-6)
        ROUND(AVG(CASE WHEN month_number <= 6
                       THEN days_past_due END), 4)   AS dpd_early,

        -- Mid window DPD (months 7-12)
        ROUND(AVG(CASE WHEN month_number BETWEEN 7 AND 12
                       THEN days_past_due END), 4)   AS dpd_mid,

        -- Late window DPD (months 13-24)
        ROUND(AVG(CASE WHEN month_number BETWEEN 13 AND 24
                       THEN days_past_due END), 4)   AS dpd_late,

        -- Missed payments in early vs mid window
        SUM(CASE WHEN month_number <= 6
                  AND payment_status = 'missed'
                 THEN 1 ELSE 0 END)                  AS missed_early,
        SUM(CASE WHEN month_number BETWEEN 7 AND 12
                  AND payment_status = 'missed'
                 THEN 1 ELSE 0 END)                  AS missed_mid,

        -- DPD acceleration: positive means worsening, negative means improving
        ROUND(
            AVG(CASE WHEN month_number BETWEEN 7 AND 12
                     THEN CAST(days_past_due AS DOUBLE) END) -
            AVG(CASE WHEN month_number <= 6
                     THEN CAST(days_past_due AS DOUBLE) END),
        4)                                           AS dpd_acceleration,

        -- First missed payment month
        MIN(CASE WHEN payment_status = 'missed'
                 THEN month_number END)              AS first_missed_month,

        MAX(CASE WHEN payment_status IN ('missed', 'post_default')
                 THEN 1 ELSE 0 END)                  AS ever_missed

    FROM repayment
    GROUP BY loan_id
""")

dpd_trajectory.createOrReplaceTempView("dpd_trajectory")

print(f"DPD trajectory features: {dpd_trajectory.count():,} rows "
      f"× {len(dpd_trajectory.columns)} columns")

print("\nDPD acceleration by default status:")
dpd_trajectory.join(
    labels.select("loan_id", "is_defaulted"), on="loan_id"
).groupBy("is_defaulted").agg(
    F.round(F.avg("dpd_early"), 3).alias("avg_dpd_early"),
    F.round(F.avg("dpd_mid"), 3).alias("avg_dpd_mid"),
    F.round(F.avg("dpd_acceleration"), 3).alias("avg_dpd_accel"),
    F.round(F.avg("missed_early"), 3).alias("avg_missed_early"),
    F.round(F.avg("missed_mid"), 3).alias("avg_missed_mid"),
).show()